# S2 Crawler

In [ ]:
import requests
import os
import aiohttp
import asyncio
from urllib.parse import quote


class S2_Request:
    def __init__(self, S2_API_KEY: str):
        if not S2_API_KEY:
            raise ValueError("Thiếu API key Semantic Scholar. Hãy cấu hình S2_API_KEY1, S2_API_KEY2 và S2_API_KEY3 trong .env hoặc biến môi trường.")
        self.headers = {"x-api-key": S2_API_KEY}
        self.url = "https://api.semanticscholar.org/graph/v1/paper/"
        self.query_params = {
            "fields": ",".join([
                "title",
                "abstract",
                "authors.name",
                "authors.affiliations",
                "references.title",
                "references.abstract"
            ])
        }

    def send_request(self, paperID: str):
        request_url = self.url + f"ARXIV:{paperID}"
        response = requests.get(request_url, params= self.query_params, headers= self.headers)

        if response.status_code == 200:
            res_data = response.json()
            print(res_data)
            return 200

        return response.status_code

    async def send_async_request(self, paperID: str):
        # https://api.semanticscholar.org/api-docs/graph
        request_url = self.url + f"ARXIV:{quote(paperID, safe='')}/references"
        references = []
        offset = 0
        params = {
            "fields": "paperId,title,authors,externalIds,year,publicationDate,venue",
            "limit": 1000,
        }
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=60)) as session:
            while True:
                await asyncio.sleep(1)
                params["offset"] = offset
                async with session.get(request_url, params=params, headers=self.headers) as response:
                    response.raise_for_status()
                    page = await response.json()
                references.extend(item.get("citedPaper") for item in page["data"])
                offset = page.get("next")
                if offset is None:
                    break
        return {"references": references}



In [ ]:
import os
import asyncio
import json
from pathlib import Path

import aiohttp
from dotenv import load_dotenv

load_dotenv()

REF_IDS_PATH = Path("data/arxiv_id_no_version.txt")  # Đổi thành đường dẫn file .txt trên Kaggle.
METADATA_PATH = Path("metadata.json")
BATCH_SIZE = 50  # Lưu sau mỗi 50 ID đã xử lý, bao gồm cả ID bị lỗi.
RETRY_ROUND_DELAY = 1  # Thời gian chờ cố định giữa các vòng retry (giây).

crawler1 = S2_Request(os.getenv("S2_API_KEY1"))
crawler2 = S2_Request(os.getenv("S2_API_KEY2"))
crawler3 = S2_Request(os.getenv("S2_API_KEY3"))

crawlers = [crawler1, crawler2, crawler3]
with REF_IDS_PATH.open("r", encoding="utf-8") as file:
    ref_ids = [line.strip() for line in file if line.strip()]


def save_results(results):
    # Thay file sau khi ghi xong để không làm hỏng checkpoint trước đó.
    temporary_path = METADATA_PATH.with_suffix(".json.tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)
    temporary_path.replace(METADATA_PATH)
    print(f"[SAVED] {len(results)}/{len(ref_ids)} ID -> {METADATA_PATH.resolve()}")


async def crawl_worker(crawler, assigned_ids, results, retry_ids):
    for paper_id in assigned_ids:
        try:
            data = await crawler.send_async_request(paper_id)
            results[paper_id] = data
            print(f"[OK] {paper_id}")
        except aiohttp.ClientResponseError as error:
            print(f"[ERROR] {paper_id}: ClientResponseError; {error}")
            results[paper_id] = None
            if error.status == 429 or 500 <= error.status < 600:
                retry_ids.append(paper_id)
                print(f"[RETRY-QUEUED] {paper_id}: thử lại sau khi kết thúc lượt hiện tại")
        except (aiohttp.ClientError, asyncio.TimeoutError) as error:
            message = str(error) or "Request timed out"
            print(f"[ERROR] {paper_id}: {type(error).__name__}; {message}")
            results[paper_id] = None
        await asyncio.sleep(1)


async def main():
    results = {}
    pending_ids = ref_ids
    retry_round = 0
    print(f"[OUTPUT] {METADATA_PATH.resolve()}")
    print(f"[START] {len(ref_ids)} ID; batch_size={BATCH_SIZE}")
    while pending_ids:
        retry_ids = []
        for start in range(0, len(pending_ids), BATCH_SIZE):
            batch = pending_ids[start:start + BATCH_SIZE]
            tasks = [
                crawl_worker(crawler, batch[i::len(crawlers)], results, retry_ids)
                for i, crawler in enumerate(crawlers)
            ]
            try:
                await asyncio.gather(*tasks)
            finally:
                # Lưu cả kết quả retry vừa thành công nếu ngắt giữa batch.
                save_results(results)
        pending_ids = retry_ids
        if pending_ids:
            retry_round += 1
            print(f"[RETRY] round={retry_round}; pending={len(pending_ids)}; wait={RETRY_ROUND_DELAY}s")
            await asyncio.sleep(RETRY_ROUND_DELAY)
    if not ref_ids:
        save_results(results)
    print("[DONE] Không còn ID lỗi 429/5xx cần retry.")
    return results


results = await main()
